# 20 · Phase 15 — Explainability: what did the agent choose, and where does the network look?

1. **Selected actions** — the configuration chosen for each signal context, and how often each
   wavelet / level / band / window / fusion option is used across test recordings.
2. **Branch attention α** — how much the fused representation relies on T, S and W.
3. **Temporal attention vs cardiac phase** — attention mass of the network's temporal-attention
   pooling over S1 / systole / S2 / diastole (TSV annotations, never seen in training), normalised
   by each phase's share of time: > 1 means the phase is attended more than its duration implies.
4. **Representative PCG regions** — attention overlaid on murmur and normal recordings.

In [ ]:
%run common/00_config.ipynb
%run common/01_signal_lib.ipynb
%run common/02_rl_lib.ipynb
%run common/03_deep_lib.ipynb
import matplotlib.pyplot as plt
import seaborn as sns
import torch

device = torch.device("cuda")
space = ConfigSpace(CFG)
meta, signals = load_deep_data(P)
qc = pd.read_csv(P["metadata"] / "records_qc.csv").set_index("record_id")
proposed = load_json(P["results"] / "proposed_model.json")["proposed_model"]
pol = load_json(P["results"] / "policies" / ("rl_joint_refined_s0.json" if proposed == "R4_full_rl_joint_refined" else "rl_full.json"))
K = pol["K"]
configs = policy_configs(meta, space, np.array(pol["policy"]), K)
te = np.where(meta.split == "test")[0]
fs = CFG["preprocessing"]["fs"]
prob = RLProblem(CFG, P, K=K)
prob.describe_policy(np.array(pol["policy"]))

In [ ]:
sel = pd.DataFrame([{**{k: str(v) for k, v in configs[i].items()}, "location": meta.location[i], "murmur": meta.murmur_label[i]} for i in te])
fig, ax = plt.subplots(1, 5, figsize=(24, 3.6))
for a, comp in zip(ax, ["wavelet", "level", "band", "window_s", "fusion"]):
    pd.crosstab(sel[comp], sel.location).plot.bar(stacked=True, ax=a, legend=a is ax[0])
    a.set_title(f"selected {comp} (test recordings)"); a.tick_params(axis="x", rotation=30)
plt.tight_layout(); savefig(fig, "20_selected_actions"); plt.show()

In [ ]:
model = load_model(proposed, 0, CFG, P).to(device)
store = RecordStore(signals, configs, fs, CFG["deep"])
probs, alphas = predict(model, store, te, device, details=True)
al = pd.DataFrame([{"record_id": meta.record_id[i], "context": meta[f"ctx_K{K}"][i], "fusion": configs[i]["fusion"],
                    "murmur": meta.murmur_label[i], "α_T": alphas[i][0], "α_S": alphas[i][1], "α_W": alphas[i][2]} for i in te])
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
al.groupby("fusion")[["α_T", "α_S", "α_W"]].mean().plot.bar(stacked=True, ax=ax[0]); ax[0].set_title("mean branch attention by selected fusion mode")
al.groupby("murmur")[["α_T", "α_S", "α_W"]].mean().plot.bar(stacked=True, ax=ax[1]); ax[1].set_title("mean branch attention by label")
plt.tight_layout(); savefig(fig, "20_branch_attention"); plt.show()
al.groupby(["context", "fusion"])[["α_T", "α_S", "α_W"]].mean().round(3)

In [ ]:
def phase_labels(rid, n):
    row = qc.loc[rid]
    seg = read_tsv(P["raw"] / row.tsv_path)
    span = annotated_span(seg)
    off = span[0] if span and span[1] - span[0] >= CFG["preprocessing"]["min_duration_s"] else 0.0
    t = np.arange(n) / fs + off
    lab = np.zeros(n, int)
    for a, b, l in seg:
        lab[(t >= a) & (t < b)] = int(l)
    return lab


@torch.no_grad()
def attention_trace(i, branch):
    n = store.win(i)
    L = len(store.x[i])
    starts = np.arange(0, max(1, L - n + 1), n)[:12]
    batch = make_batch(store, [(i, int(s)) for s in starts], device, model.branches)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        _, det = model(batch, return_details=True)
    w = det["time_attention"][branch].float().cpu().numpy()
    trace = np.zeros(L)
    for s, wi in zip(starts, w):
        trace[s: s + n] = np.interp(np.linspace(0, len(wi) - 1, min(n, L - s)), np.arange(len(wi)), wi) * len(wi)
    return trace, (starts[0], starts[-1] + n)


rows = []
for i in te:
    for b in [bb for bb in ("T", "S", "W") if bb in configs[i]["fusion"]]:
        tr, (a0, a1) = attention_trace(i, b)
        lab = phase_labels(meta.record_id[i], len(store.x[i]))[a0:a1]
        tr = tr[a0:a1]
        tot = tr.sum()
        for ph, nm in ((1, "S1"), (2, "systole"), (3, "S2"), (4, "diastole")):
            m = lab == ph
            if m.any() and tot > 0:
                rows.append({"branch": b, "murmur": meta.murmur_label[i], "phase": nm, "attention_ratio": (tr[m].sum() / tot) / m.mean()})
ph = pd.DataFrame(rows)
ph.to_csv(P["tables"] / "20_attention_by_phase.csv", index=False)
g = sns.catplot(data=ph, x="phase", y="attention_ratio", hue="murmur", col="branch", kind="bar", height=3.6, aspect=1.2, errorbar=("ci", 95))
g.refline(y=1, color="k", ls=":")
g.fig.suptitle("Temporal attention per cardiac phase ÷ phase duration share (test)", y=1.03)
savefig(g.fig, "20_attention_by_phase"); plt.show()
ph.groupby(["branch", "murmur", "phase"]).attention_ratio.mean().unstack().round(2)

In [ ]:
pt = pd.Series(probs, index=te)
tp = [i for i in te if meta.y[i] == 1 and "T" in configs[i]["fusion"]]
tn = [i for i in te if meta.y[i] == 0 and "T" in configs[i]["fusion"]]
picks = [max(tp, key=lambda i: pt[i]), min(tn, key=lambda i: pt[i])] if tp and tn else te[:2]
cfg_label = lambda c: f"{c['wavelet']}-L{c['level']}-{c['band'][0]}-{c['band'][1]}Hz-{c['window_s']}s-{c['fusion']}"
colors = {1: "tab:red", 2: "tab:orange", 3: "tab:green", 4: "tab:blue"}
fig, ax = plt.subplots(len(picks), 1, figsize=(16, 3.4 * len(picks)))
for a, i in zip(np.atleast_1d(ax), picks):
    tr, _ = attention_trace(i, "T")
    x = store.x[i]
    lab = phase_labels(meta.record_id[i], len(x))
    n = min(len(x), 4 * fs)
    t = np.arange(n) / fs
    a.plot(t, x[:n] / np.abs(x[:n]).max(), lw=0.4, color="gray")
    a.plot(t, tr[:n] / (tr[:n].max() + 1e-9), color="k", lw=1.5, label="temporal attention (T branch)")
    for ph_, c in colors.items():
        a.fill_between(t, -1, 1, where=lab[:n] == ph_, color=c, alpha=0.12, step="mid")
    r = qc.loc[meta.record_id[i]]
    a.set_title(f"{meta.record_id[i]} | label {meta.murmur_label[i]} ({r.sys_timing}, {r.sys_grading}) | p = {pt[i]:.2f} | config {cfg_label(configs[i])}", fontsize=9)
    a.legend(loc="upper right", fontsize=8)
np.atleast_1d(ax)[-1].set_xlabel("time (s)   shading: S1 red, systole orange, S2 green, diastole blue")
plt.tight_layout(); savefig(fig, "20_representative_regions"); plt.show()